In [2]:
import xarray as xr
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [1]:
msg_file = '/mnt/data8tb/fire_detection/msg/L1b/MSG3-SEVI-MSG15-0100-NA-20230930171241.955000000Z-NA.nat'
msg_cloud = '/mnt/data8tb/fire_detection/msg/L1b/MSG3-SEVI-MSGCLMK-0100-0100-20230930171500.000000000Z-NA.grb'
modis_file = '/mnt/data8tb/fire_detection/modis/MOD/L1b/MOD021KM.A2024165.1110.061.2024165191849.hdf'
modis_cloud = '/mnt/data8tb/fire_detection/modis/MOD/CM/MOD35_L2.A2023273.1105.061.2023273191143.hdf'


In [3]:
MODIS_WAVELENGTHS = {
    "1": 0.645, 
    "2": 0.8585,
    "3": 0.469,
    "4": 0.555,
    "5": 1.24,
    "6": 1.64,
    "7": 2.13,
    "8": 0.4125,
    "9": 0.443,
    "10": 0.488,
    "11": 0.531,
    "12": 0.551,
    "13lo": 0.667,
    "13hi": 0.667,
    "14lo": 0.678,
    "14hi": 0.678,
    "15": 0.748,
    "16": 0.8695,
    "17": 0.905,
    "18": 0.936,
    "19": 0.940,
    "20": 3.75,
    "21": 3.959,
    "22": 3.959,
    "23": 4.05,
    "24": 4.4655,
    "25": 4.5155,
    "26": 1.375,
    "27": 6.715,
    "28": 7.325,
    "29": 8.55,
    "30": 9.73,
    "31": 11.03,
    "32": 12.02,
    "33": 13.335,
    "34": 13.635,
    "35": 13.935,
    "36": 14.235,
}

In [4]:
# MODIS Radiances
file = [modis_file]
scn = Scene(
            reader="modis_l1b",
            filenames=file
        )

# Load radiance bands
# channels = get_modis_channel_numbers() # This assigns the wrong order of bands
channels = list(MODIS_WAVELENGTHS.keys())
scn.load(channels, generate=False, calibration='radiance')

In [7]:
from pyresample import create_area_def

area_def = create_area_def('my_area',
                           {'proj': 'longlat', 'datum': 'WGS84'},
                           area_extent=[-10.019531, 30.22889, 46.617188, 49.012224],
                           resolution=0.027,
                           units='degrees',
                           description='Gloxx   bal degree lat-lon grid')

Rounding shape to (696, 2098) and resolution from (0.027000000000001023, 0.027000000000001023) meters to (0.02699557626310772, 0.02698754885057472) meters


In [8]:
new_scn = scn.resample(area_def)

In [9]:
ds_resampled = new_scn.to_xarray_dataset()

In [10]:
ds_resampled

<xarray.Dataset> Size: 222MB
Dimensions:  (y: 696, x: 2098)
Coordinates:
    crs      object 8B GEOGCRS["unknown",DATUM["World Geodetic System 1984",E...
  * y        (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.32 30.3 30.27 30.24
  * x        (x) float64 17kB -10.01 -9.979 -9.952 -9.925 ... 46.55 46.58 46.6
Data variables: (12/38)
    14hi     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    9        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    3        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    15       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    22       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    14lo     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    ...       ...
    28       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    13lo     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    8        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    13hi     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    16       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    23       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
Attributes: (12/15)
    calibration:          radiance
    coordinates:          ('longitude', 'latitude')
    modifiers:            ()
    standard_name:        toa_outgoing_radiance_per_unit_wavelength
    platform_name:        EOS-Terra
    resolution:           1000
    ...                   ...
    start_time:           2024-06-13 11:10:00
    sensor:               modis
    units:                Watts/m^2/micrometer/steradian
    end_time:             2024-06-13 11:15:00
    ancillary_variables:  []
    reader:               modis_l1b

In [11]:
# MODIS cloud mask
scn = Scene(
            reader="modis_l2",
            filenames=[modis_cloud]
        )
# Load cloud mask data
datasets = scn.available_dataset_names()
# Needs to be loaded at 1000 m resolution for all channels to match
scn.load(datasets, generate=False, resolution=1000) 

In [12]:
new_scn_cloud = scn.resample(area_def)

In [13]:
ds_resampled_clouds = new_scn_cloud.to_xarray_dataset()

In [16]:
ds_resampled_clouds

<xarray.Dataset> Size: 3MB
Dimensions:            (y: 696, x: 2098)
Coordinates:
    crs                object 8B GEOGCRS["unknown",DATUM["World Geodetic Syst...
  * y                  (y) float64 6kB 49.0 48.97 48.94 ... 30.3 30.27 30.24
  * x                  (x) float64 17kB -10.01 -9.979 -9.952 ... 46.58 46.6
Data variables:
    quality_assurance  (y, x) uint8 1MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    cloud_mask         (y, x) uint8 1MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
Attributes:
    modifiers:            ()
    platform_name:        EOS-Terra
    resolution:           1000
    rows_per_scan:        10
    start_time:           2023-09-30 11:05:00
    sensor:               modis
    end_time:             2023-09-30 11:10:00
    ancillary_variables:  []
    reader:               modis_l2

In [17]:
ds_resampled

<xarray.Dataset> Size: 222MB
Dimensions:  (y: 696, x: 2098)
Coordinates:
    crs      object 8B GEOGCRS["unknown",DATUM["World Geodetic System 1984",E...
  * y        (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.32 30.3 30.27 30.24
  * x        (x) float64 17kB -10.01 -9.979 -9.952 -9.925 ... 46.55 46.58 46.6
Data variables: (12/38)
    14hi     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    9        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    3        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    15       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    22       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    14lo     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    ...       ...
    28       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    13lo     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    8        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    13hi     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    16       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    23       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
Attributes: (12/15)
    calibration:          radiance
    coordinates:          ('longitude', 'latitude')
    modifiers:            ()
    standard_name:        toa_outgoing_radiance_per_unit_wavelength
    platform_name:        EOS-Terra
    resolution:           1000
    ...                   ...
    start_time:           2024-06-13 11:10:00
    sensor:               modis
    units:                Watts/m^2/micrometer/steradian
    end_time:             2024-06-13 11:15:00
    ancillary_variables:  []
    reader:               modis_l1b